In [1]:
# =========================================================
# IMPORT NECESSARY LIBRARIES
# =========================================================
# Import NetworkX for creating and manipulating complex networks (graphs)
import networkx as nx

# Import Matplotlib for plotting and visualization
import matplotlib.pyplot as plt

# Import random for stochastic (probabilistic) processes
import random

# Import NumPy for numerical computations (e.g., averages)
import numpy as np

# Import imageio for creating animated GIFs from images
import imageio

# Import os for file handling (deleting temporary files)
import os

# Import matplotlib utilities for color normalization and colormaps
import matplotlib.colors as mcolors
import matplotlib.cm as cm


# =========================================================
# CONFIGURATION & GLOBAL PARAMETERS
# =========================================================

# Total number of people (nodes of type 'person') in the network
NUM_PEOPLE = 100

# Number of evolutionary iterations (time steps)
ITERATIONS = 10

# Names of social foci (clubs/places people can join)
FOCI_NAMES = ['Gym', 'Eatout', 'Movie Club', 'Karate Club', 'Yoga Club']

# List to store filenames of frames for GIF generation
frames = []


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def get_person_nodes(G):
    """
    Returns a list of all nodes marked as 'person'.
    These represent individuals in the social network.
    """
    return [n for n, attr in G.nodes(data=True) if attr['type'] == 'person']


def get_foci_nodes(G):
    """
    Returns a list of all nodes marked as 'focus'.
    These represent social environments or activity hubs.
    """
    return [n for n, attr in G.nodes(data=True) if attr['type'] == 'focus']


def visualize(G, title, iteration_name):
    """
    Visualizes the graph with:
    - A BMI heatmap for people nodes
    - A distinct color and shape for foci nodes
    - Edge transparency for readability
    """
    # Create a new figure for plotting
    plt.figure(figsize=(12, 10))

    # Compute node positions using a force-directed (spring) layout
    # Fixed seed ensures consistent layout across iterations
    pos = nx.spring_layout(G, k=0.15, seed=42)

    # Separate people nodes and foci nodes
    pnodes = get_person_nodes(G)
    fnodes = get_foci_nodes(G)

    # Extract BMI values for people nodes
    bmis = [G.nodes[n]['val'] for n in pnodes]

    # --- 1. Setup BMI Heatmap for People Nodes ---

    # Choose a reversed Red-Yellow-Blue colormap
    # Blue = low BMI, Red = high BMI
    cmap = cm.get_cmap('RdYlBu_r')

    # Normalize BMI values to range [0, 1] for color mapping
    # BMI assumed between 15 and 40
    norm = mcolors.Normalize(vmin=15, vmax=40)

    # Assign a color to each person node based on BMI
    p_colors = [cmap(norm(bmi)) for bmi in bmis]

    # Scale node sizes proportionally to BMI
    p_sizes = [bmi * 15 for bmi in bmis]

    # Draw person nodes with heatmap colors and scaled sizes
    nodes_p = nx.draw_networkx_nodes(
        G, pos,
        nodelist=pnodes,
        node_color=p_colors,
        node_size=p_sizes,
        edgecolors='black',
        alpha=0.9
    )

    # --- 2. Draw Foci Nodes with a Distinct Color ---

    # Draw foci nodes as indigo-colored squares
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=fnodes,
        node_color='indigo',
        node_size=1200,
        node_shape='s',
        label='Social Foci'
    )

    # Prepare labels for people (node id) and foci (name)
    p_labels = {n: str(n) for n in pnodes}
    f_labels = {n: n for n in fnodes}

    # Draw labels on the graph
    nx.draw_networkx_labels(G, pos, labels=p_labels, font_size=8, font_color='black')
    nx.draw_networkx_labels(G, pos, labels=f_labels, font_color='white', font_weight='bold')

    # Draw edges with low opacity to reduce clutter
    nx.draw_networkx_edges(G, pos, alpha=0.2)

    # --- Add Colorbar for BMI Heatmap ---

    # Create a ScalarMappable for the colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Add colorbar to the current axis
    cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', fraction=0.046, pad=0.04)
    cbar.set_label('BMI Value', rotation=270, labelpad=15)

    # Add plot title with iteration information
    plt.title(f"{title}\n({iteration_name})", fontsize=14)

    # Save current frame as an image file
    filename = f"step_{len(frames)}.png"
    plt.savefig(filename)

    # Store filename for GIF creation
    frames.append(filename)

    # Display the plot
    plt.show()


# =========================================================
# INITIALIZATION
# =========================================================

print("Initializing Network...")

# Create an undirected graph
G = nx.Graph()

# 1. Adding People Nodes
for i in range(NUM_PEOPLE):
    # Assign each person a random BMI between 15 and 40
    # 'type' distinguishes people from foci
    bmi = random.randint(15, 40)
    G.add_node(i, type='person', val=bmi)

# Visualize after adding people nodes
visualize(G, "Step 1: Adding 100 People Nodes", "Init")


# 2. Adding Foci Nodes
for name in FOCI_NAMES:
    # Store lowercase name (without spaces) for logic consistency
    G.add_node(name, type='focus', name=name.lower().replace(" ", ""))

# Visualize after adding foci nodes
visualize(G, "Step 2: Adding 5 Social Foci Nodes", "Init")


# 3. Initial Random Edges (Connecting people to foci)
print("Assigning initial memberships...")

for i in range(NUM_PEOPLE):
    # Each person joins 1 or 2 randomly selected foci
    for club in random.sample(FOCI_NAMES, random.randint(1, 2)):
        G.add_edge(i, club)

# Visualize initial membership connections
visualize(G, "Step 3: Initial Membership Edges", "Init")


# =========================================================
# EVOLUTIONARY MECHANISMS
# =========================================================

def homophily(G):
    """
    Implements Homophily (Selection Effect).
    People with similar BMI values are more likely to connect.

    Probability formula:
    p = 1 / (diff + 1000)

    where diff = |BMI_u - BMI_v|
    """
    pnodes = get_person_nodes(G)
    for u in pnodes:
        for v in pnodes:
            if u != v and not G.has_edge(u, v):
                # Absolute difference in BMI
                diff = abs(G.nodes[u]['val'] - G.nodes[v]['val'])

                # Connection probability decreases with BMI difference
                p = 1.0 / (diff + 1000)

                # Add edge probabilistically
                if random.random() < p:
                    G.add_edge(u, v)


def triadic_closure(G):
    """
    Implements Triadic Closure.
    If A-B and B-C exist, A-C becomes more likely.

    Probability formula:
    Pr(A-C) = 1 - (1 - p)^k

    where:
    p = base probability
    k = number of common people neighbors
    """
    p = 0.01
    pnodes = get_person_nodes(G)

    for i in range(len(pnodes)):
        for j in range(i + 1, len(pnodes)):
            u, v = pnodes[i], pnodes[j]
            if not G.has_edge(u, v):
                # Count common neighbors that are people
                common = [
                    n for n in nx.common_neighbors(G, u, v)
                    if G.nodes[n]['type'] == 'person'
                ]
                k = len(common)
                if k > 0:
                    # Apply triadic closure probability
                    if random.random() < (1 - (1 - p) ** k):
                        G.add_edge(u, v)


def foci_closure(G):
    """
    Implements Foci Closure.
    Two people become friends because they share common foci.

    Probability formula:
    Pr(A-B) = 1 - (1 - p)^k

    where:
    k = number of shared foci
    """
    p = 0.01
    pnodes = get_person_nodes(G)

    for i in range(len(pnodes)):
        for j in range(i + 1, len(pnodes)):
            u, v = pnodes[i], pnodes[j]
            if not G.has_edge(u, v):
                # Count shared foci neighbors
                common = [
                    n for n in nx.common_neighbors(G, u, v)
                    if G.nodes[n]['type'] == 'focus'
                ]
                k = len(common)
                if k > 0:
                    if random.random() < (1 - (1 - p) ** k):
                        G.add_edge(u, v)


def membership_closure(G):
    """
    Implements Membership Closure.
    A person joins a focus because their friends are members.

    Probability formula:
    Pr(P-F) = 1 - (1 - p)^k

    where:
    k = number of friends already in that focus
    """
    p = 0.01
    pnodes = get_person_nodes(G)
    fnodes = get_foci_nodes(G)

    for u in pnodes:
        for f in fnodes:
            if not G.has_edge(u, f):
                # Friends of person u
                friends_of_u = set(G.neighbors(u))

                # Members of focus f
                members_of_f = set(G.neighbors(f))

                # k = common friends who are members of the focus
                k = len(friends_of_u.intersection(members_of_f))
                if k > 0:
                    if random.random() < (1 - (1 - p) ** k):
                        G.add_edge(u, f)


def social_influence(G):
    """
    Implements Social Influence (Peer Effect).

    BMI update rules:
    - Eatout focus -> BMI increases by 1 (up to max 40)
    - Gym/Karate/Yoga -> BMI decreases by 1 (down to min 15)
    """
    fnodes = get_foci_nodes(G)

    for f in fnodes:
        f_name = G.nodes[f]['name']
        for neighbor in G.neighbors(f):
            # Apply influence only to people nodes
            if G.nodes[neighbor]['type'] == 'person':
                # Unhealthy influence
                if f_name == 'eatout' and G.nodes[neighbor]['val'] < 40:
                    G.nodes[neighbor]['val'] += 1
                # Healthy influence
                elif f_name in ['gym', 'karateclub', 'yogaclub'] and G.nodes[neighbor]['val'] > 15:
                    G.nodes[neighbor]['val'] -= 1

# =========================================================
# RUNNING THE MAIN SIMULATION
# =========================================================

print("\n--- Starting Evolutionary Iterations ---")

# Store average BMI over time for longitudinal analysis
avg_bmi_history = []

for i in range(1, ITERATIONS + 1):
    print(f"Simulating Iteration {i}...")

    # 1. Apply homophily mechanism
    homophily(G)
    if i == 1:
        visualize(G, "After Implementing Homophily", f"Iter {i}")

    # 2. Apply triadic closure
    triadic_closure(G)
    if i == 1:
        visualize(G, "After Triadic Closure", f"Iter {i}")

    # 3. Apply foci closure
    foci_closure(G)
    if i == 1:
        visualize(G, "After Foci Closure", f"Iter {i}")

    # 4. Apply membership closure
    membership_closure(G)
    if i == 1:
        visualize(G, "After Membership Closure", f"Iter {i}")

    # 5. Apply social influence (contagion step)
    social_influence(G)

    # Compute and record current average BMI
    current_avg_bmi = np.mean([G.nodes[n]['val'] for n in get_person_nodes(G)])
    avg_bmi_history.append(current_avg_bmi)

    # Periodic visualization every 2 iterations (after iteration 1)
    if i > 1 and i % 2 == 0:
        visualize(G, "Evolution in Progress (Mid-Simulation)", f"Iter {i}")

# Visualize final network state
visualize(G, "Final Network State", f"Iter {ITERATIONS}")

# =========================================================
# FINAL ANALYSIS & GIF GENERATION
# =========================================================

print("\nGenerating GIF...")

# Create an animated GIF from saved frames
with imageio.get_writer('fatman_evolution_heatmap.gif', mode='I', duration=1.0) as writer:
    for f in frames:
        image = imageio.imread(f)
        writer.append_data(image)
        # Remove temporary image file after adding to GIF
        os.remove(f)

print("Generating Final Analysis Plot...")

# Plot average BMI over time
plt.figure(figsize=(10, 6))
plt.plot(
    range(1, ITERATIONS + 1),
    avg_bmi_history,
    color='#d62728',
    marker='o',
    linewidth=2,
    label='Avg BMI'
)

# Plot horizontal line for initial average BMI
initial_avg = avg_bmi_history[0]
plt.axhline(
    y=initial_avg,
    color='gray',
    linestyle='--',
    label=f'Initial Avg ({initial_avg:.2f})'
)

# Fill area above and below initial average for emphasis
plt.fill_between(
    range(1, ITERATIONS + 1),
    avg_bmi_history,
    initial_avg,
    where=(np.array(avg_bmi_history) >= initial_avg),
    interpolate=True,
    color='red',
    alpha=0.1
)
plt.fill_between(
    range(1, ITERATIONS + 1),
    avg_bmi_history,
    initial_avg,
    where=(np.array(avg_bmi_history) < initial_avg),
    interpolate=True,
    color='blue',
    alpha=0.1
)

# Add plot labels and styling
plt.title("Obesity Contagion Analysis: Longitudinal BMI Trend", fontsize=14)
plt.xlabel("Iteration (Time Step)")
plt.ylabel("Average Population BMI")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)

# Display the plot
plt.show()

print("\nSimulation Complete!")
print("The network evolution is saved as 'fatman_evolution_heatmap.gif'.")
print("The final BMI trend plot has been displayed.")

Output hidden; open in https://colab.research.google.com to view.